In [ ]:
# Purpose - plots fields and cuts for a single file

findex = 252 #232 #-7 #270 #260 #240 #25 #180 #144 #26 #9 #100 #140 #2 #160 #2 #140 #140 

step_choice = "all" #"penultimate" #"last" #"last" #all

In [ ]:
# Imports and Folder Selection

%load_ext autoreload
%autoreload 2

from graphet import Data
from graphet.plugins import TristanV2 
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
import sys
sys.path.append('../../v2_proc_files')
from useful_funcs import *
from TristanRun import TristanRun
from FieldLineHelper import FieldLineHelper 

plt.close('all')

%matplotlib widget

set_standard_plot_params()

# Read Input File Data

run = TristanRun(findex)
gyrofreq = run.gyrofreq

cc = run.cc
B0 = run.B0

tstep_choices = {"first": [run.steps[0]], "last": [run.steps[-1]], "all": run.steps[:-1], "penultimate": [run.steps[-2]]}

my_steps = tstep_choices[step_choice]
#my_steps = list(range(85, 681, 5))
#my_steps = [85]
#my_steps = range(200, 223)
print(len(my_steps))
#my_steps = run.steps[-6:]

#boilerplate = run.boilerplate + f" tstep = {my_steps[0]}"

In [ ]:
# Configure Data

d = run.config_data(my_steps, "fields", includes_pressure = False)

In [ ]:
#ts = np.where(my_steps == 40)[0][0]
#ts = 15

#timesteps = my_steps[7:]
timesteps = range(len(my_steps))

In [ ]:
# Choose Regions of Interest

#rect = {10: [59.1, 60.9, 162, 167], 0.3: [59, 61, 105, 135]}[run.sigma]
#rect = {10: [55, 65, 162, 168], 0.3: [59, 61, 105, 135]}[run.sigma]
#rect = {10: [59.1, 60.9, 164, 174], 0.3: [59, 61, 105, 135]}[run.sigma]
rect = [10, run.xlim - 10, 0, run.ylim]
rect = [0, run.xlim, 0, run.ylim]

xlim = rect[1]
ylim = rect[3]


nsteps = len(my_steps)

#roi = fq_cut(d.fields, ts, rect)

# Average roi over selected timesteps
rois = [fq_cut(d.fields, ts, rect) for ts in timesteps]
roi = sum(rois) / len(rois)


#for i in range(1, nsteps):
#    roi += fq_cut(d.fields, i, rect)

#roi /= nsteps

#xcut_y = 164.4
xcut_y = (rect[2] + rect[3]) / 2 #5950
ycut_x = (rect[0] + rect[1]) / 2 
#xcut_y = run.ylim / 2 + 20
xcut_y = 5950
xcut_y = run.ylim / 2

xstrip = fq_xcut(roi, xcut_y)
ystrip = fq_ycut(roi, ycut_x)

In [ ]:
print(list(d.fields.keys()))

In [ ]:
# Plot Regions

cs = roi

#ez_treal = cs.bx * cs.vely - cs.by * cs.velx 
#ez_nontreal = cs.ez - ez_treal 
ez_tot = cs.ez
JdotE = cs.ex * cs.jx + cs.ey * cs.jy + cs.ez * cs.jz

Az = compute_Az(cs)

cs['ez_nonideal'] = cs.ez + cs.velx * cs.by - cs.vely * cs.bx
#cs['ez_ideal'] = cs.ez - cs.ez_nonideal

dens = cs.dens1 + cs.dens2

esq, bsq, chi = sqrd(cs) 

adj_factor = run.beta_A

bsq_adj =  adj_factor ** 2 * bsq
chi_adj = (bsq_adj - esq) / (bsq_adj + esq)

j_cross_b = cs.jz * cs.bx - cs.jx * cs.bz

cs['lsigma'] = cs.pb / cs.dens

v_A = run.beta_A
print(run.beta_A)

plt.figure()

# --------------STREAMLINES--------------

plot_streamline(plt, cs, cs.bx, cs.by, 0.6, 'white') 
#plot_streamline(plt, cs, cs.velx, cs.vely, 6, 'yellow') 

# --------------CONTOURS--------------

#plt.contour(cs.x, cs.y, chi_adj, [0], colors = 'green')

Ainc = (0.16 * run.beta_A * run.cc) * run.B0 * run.output_interval

#plt.contour(cs.x, cs.y, Az, np.linspace(np.min(Az), np.max(Az), 10) , colors = 'white', linewidths = 0.6, linestyles = 'solid') # '#00ff00')
#plt.contour(cs.x, cs.y, Az_bad, [1, 1 + Ainc * 500], colors = 'orange')
#plt.contour(cs.x, cs.y, cs.dens, [2.5], colors = 'green')
# --------------AREA PLOTS--------------
# SCALARS

#plot_fq3(cs, cs.dens, vabsmax = 30, cmap= 'viridis', positive = True)

print("Average density:", cs.dens.mean().compute())

#plot_fq3(cs, cs.lsigma, vabsmax = 1.5, cmap= 'viridis', positive = True)
#plot_fq3(cs, cs.dens - 0.5, vabsmax = 3.5 - 0.5, cmap = 'viridis', positive = True)
#plot_fq3(cs, cs.ndens, vabsmax = 3.5, cmap = 'viridis', positive = True)
#plot_fq3(cs, cs.epsilon - 1, vabsmax = 0.008, positive = True)
#plot_fq3(cs, (cs.px + cs.py) / 2, vabsmax = 2, positive = True)

# FIELDS

#plot_fq3(cs, cs.ex, vabsmax = 0.05, cmap = 'seismic')
#plot_fq3(cs, cs.ey / B0, "$E_y / B_0$", vabsmax = 0.3)
#plot_fq3(cs, cs.ez, cbar_label = "$E_z$", vabsmax = 1.5 * (0.1 * run.beta_A), cmap = 'seismic')


#plot_fq3(cs, cs.by, cbar_label = "$B_y / B_0$", vabsmax = 0.15, cmap = 'seismic')

#plot_fq3(cs, j_cross_b / (cs.epsilon * cs.dens), cbar_label = "$F_y$", vabsmax = 0.01, positive = True)

#plot_fq3(cs, cs.by, vabsmax = 0.15)
#plot_fq3(cs, Az)

#plot_fq3(cs, cs.jz, cbar_label = 'Jz', vabsmax = 0.10, cmap = 'seismic')

# Per-species current, from the momentum-density (stress-energy T^{0i}) field components.
# T0D_s is the mass-weighted momentum density of species s in direction D -- the same raw
# quantity "dens_s" is for mass density -- so T0D_s / (ppc * mass_s) is species s's number
# flux in direction D (same normalization as ndens_s = dens_s / (ppc * mass_s)); multiplying
# by the species charge gives its current. Checked numerically against jx/jz: correct sign
# and spatial structure (corr ~0.93-0.95), but NOT on the same absolute scale as jx/jy/jz --
# Tristan's 2D3V scheme deposits in-plane (x,y) current via a charge-conserving scheme while
# out-of-plane (z) current is a simpler direct deposit, so x/y and z pick up different,
# un-recovered calibration factors here. Treat these as relative/qualitative, re-tune vabsmax.

q_electron, q_ion = -1, 1

#plot_fq3(cs, q_electron * cs.T0X1 / run.ppc, cbar_label = 'J_ex', vabsmax = 0.04, cmap = 'seismic')
#plot_fq3(cs, q_electron * cs.T0Y1 / run.ppc, cbar_label = 'J_ey', vabsmax = 0.04, cmap = 'seismic')
#plot_fq3(cs, q_electron * cs.T0Z1 / run.ppc, cbar_label = 'J_ez', vabsmax = 0.04, cmap = 'seismic')
#plot_fq3(cs, q_ion * cs.T0X2 / (run.ppc * run.mime), cbar_label = 'J_ix', vabsmax = 0.04, cmap = 'seismic')
#plot_fq3(cs, q_ion * cs.T0Y2 / (run.ppc * run.mime), cbar_label = 'J_iy', vabsmax = 0.04, cmap = 'seismic')
plot_fq3(cs, q_ion * cs.T0Z2 / (run.ppc * run.mime), cbar_label = 'J_iz', vabsmax = 0.04, cmap = 'seismic')
#plot_fq3(cs, (cs.jz / j0) / cs.dens, cbar_label = 'Jz / n', vabsmax = 0.4, positive = True)

# ENERGY

#plot_fq3(cs, (cs.ex * cs.jx) / (j0 * run.B0), cbar_label = '$J_x E_x$', vabsmax = 1e-2, cmap = 'seismic')
#plot_fq3(cs, (cs.ey * cs.jy) / (j0 * run.B0), cbar_label = '$J_y E_y$', vabsmax = 1e-2, cmap = 'seismic')
#plot_fq3(cs, (cs.jz * run.c_omp / run.B0) * cs.ez / run.B0, cbar_label = '$J_z E_z$', vabsmax = 1e-3, cmap = 'seismic')
#plot_fq3(cs, (cs.ex * cs.jx + cs.ey * cs.jy + cs.ez * cs.jz) / (j0 * run.B0), cbar_label = '$J_z E_z$', vabsmax = 1e-2, cmap = 'seismic')

# NONIDEAL EFFECTS

#plot_fq3(cs, cs.ez, vabsmax = 0.15)
#plot_fq3(cs, cs.ez_ideal / run.B0, cmap = "seismic", vabsmax = 0.1)
#plot_fq3(cs, cs.ez_nonideal / run.B0, cmap = "seismic", vabsmax = 0.1)
#plot_fq3(cs, cs.jz, f"$J_z$", vabsmax = 1e-4, cmap = 'seismic')
#plot_fq3(cs, cs.jz * cs.ez, cbar_label = "JzEz", vabsmax = 1e-3, cmap = 'seismic')
#plot_fq3(cs, cs.ez_nonideal / cs.jz, vabsmax = 0.1)

# VELOCITY

#plot_fq3(cs, cs.velx / v_A, cbar_label = "$v_x / v_A$", vabsmax = 0.6, cmap = 'seismic')
#plot_fq3(cs, cs.velx / v_A, vabsmax = 0.2, cmap = 'seismic')
#plot_fq3(cs, cs.velz / run.beta_A, cbar_label = '$v_y / v_A$', vabsmax = 1, cmap = 'seismic')
#plot_fq3(cs, cs.dens * cs.velx / v_A, vabsmax = 0.2, cmap = 'seismic')

# --------------PLASMOID IDENTIFIERS--------------

# Find points where ystrip.bx switches sign
sign_change_indices = np.where(np.diff(np.sign(ystrip.bx)) > 0)[0]

ts = -1
#plt.title(run.boilerplate + f"\n t = {run.steps[ts]} $\sigma =$ {run.sigma}")
#plt.vlines(ycut_x, rect[2], rect[3], colors = 'yellow', linestyle = '--')
#plt.hlines(xcut_y, rect[0], rect[1], colors = 'yellow', linestyle = '--')
plt.xlim(rect[:2])
plt.ylim(rect[2:])
#save_polished_fig('dpp_s0.3.png')
plt.show()

In [ ]:
print(0 / 0)

In [ ]:
# APS intro slide plot

fig, ax = plt.subplots(1, 1, figsize=(7, 5))

flh1 = None
rel_ts = 0

large_field_downsampling = 2      #2
large_contour_downsampling = 4    # 4
large_num_contours = 8            # 8

small_field_downsampling = 1      #1
small_contour_downsampling = 1    #1
left_num_contours = 8             #8
right_num_contours = 3

buffer = 10
split = 1
dens_cap = 3.5
cmap = 'inferno'

roi = fq_cut(d.fields, rel_ts, [0 + buffer, (run.xlim - buffer) / split, 0 + buffer, (run.ylim - buffer) / split])
ez_rep = 0

im = plot_fq3_flipped(roi, roi.dens, plot=ax, vabsmax=dens_cap, positive=True, downsample=large_field_downsampling, cmap=cmap)
if flh1 is None:
    flh1 = FieldLineHelper(run, large_num_contours, contour_ds=large_contour_downsampling)
flh1.plot_field_lines(roi, ez_rep, plot=ax, arrow_spacing=-1, linewidth=0.5, transpose=True)

# remove axes frame / spines and any figure edge
for sp in ax.spines.values():
    sp.set_visible(False)
fig.patch.set_edgecolor('none')
fig.patch.set_linewidth(0)

# ensure no axis decorations or margins
ax.set_axis_off()
plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
ax.margins(0)

ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])

ax.set_ylim(buffer, run.xlim - buffer)

#save_polished_fig('aps_title_slide_picture', dpi = 500)

# add a vertical colorbar to the right of the image
# shrink main axes to make room and add colorbar to the right without overlap
box = ax.get_position()  # Bbox: (x0, y0, x1, y1)
new_width = box.width * 0.88  # leave ~12% of axes width for the colorbar area
ax.set_position([box.x0, box.y0, new_width, box.height])

# place colorbar to the right of the resized axes
cax = fig.add_axes([box.x0 + new_width + 0.02, box.y0, 0.02, box.height])
cb = fig.colorbar(im, cax=cax, orientation='vertical')
cb.set_label('$n / n_0$')



plt.show()

In [ ]:
# Plot only the colorbar from the last image `im` with label
fig_cb_only, ax_cb_only = plt.subplots(figsize=(1.6, 5))
ax_cb_only.set_axis_off()

# dedicated axes for the colorbar (left, bottom, width, height)
cax = fig_cb_only.add_axes([0.35, 0.05, 0.25, 0.9])
cb_only = fig_cb_only.colorbar(im, cax=cax, orientation='vertical')
#cb_only.set_label('$n / n_0$')

plt.show()

In [ ]:
# Y-strip 

strip = ystrip

strip['ez_ideal'] = strip.bx * strip.vely - strip.by * strip.velx 
strip['ez_nonideal'] = strip.ez - strip.ez_ideal 
ez_tot = strip.ez

JdotE = strip.jx * strip.ex + strip.jy * strip.ey + strip.jz * strip.ez
dens = strip.dens1 + strip.dens2

esq, bsq, chi = sqrd(strip)

adj_factor = run.beta_A ** 2


bsq_adj = adj_factor ** 2 * bsq
chi_adj = (bsq_adj - esq) / (bsq_adj + esq)

sign_change = np.diff(np.sign(chi_adj))
#y_gate_index = np.where(sign_change > 0)[-1][-1]

jadj = 200

plt.figure()
plt.title(run.boilerplate + f"\n t = {run.steps[ts]}, x = {ycut_x}")
#plt.plot(strip.y, ez_nontreal / (cc * B0), label = "ez_nontreal / c B_0")
#plt.plot(strip.y, ez_treal / (cc * B0), label = "ez_treal / c B_0")
#plt.plot(strip.y, strip.ez_nonideal, label = "$E_{z, ni} / c B_0$", color = '#1f77b4') #ez
#plt.plot(strip.y, bsq / B0 ** 2, label = "bsq / B0^2") #bsq
#plt.plot(strip.y, strip.jz, label = '$J_z (c / \omega_p) / B_0$', color = '#2ca02c') #jz
#plt.plot(strip.y, strip.bx / B0, label = "$B_x / B_0$", color = '#ff7f0e')
#plt.plot(strip.y, strip.bz / B0, label = "Bz / B0")
#plt.plot(strip.y, chi, label = '$\chi$')
#plt.plot(strip.y, chi_adj, label = '$\chi_{adj}$', color = 'orange')
#plt.plot(strip.y, strip.vely / run.beta_A, label = "$v_y / v_A$", color = '#9467bd')
#plt.plot(strip.y, strip.velx / run.beta_A, label = "$v_y / v_A$", color = 'red')
plt.plot(strip.y, strip.dens / 20, label = '$n / 20$')
#plt.plot(strip.y, (strip.vely / v_A) / (strip.bx / run.B0), label = "$v_y / v_A$ / $(B_x / B_0)$", color = '#8c564b')

plt.plot(strip.y, strip.ez_nonideal / strip.jz, label = r"$\eta = E_{z, ni} / J_z$", color = 'orange')


avg_velx_v_A = np.mean(strip.velx.values / run.beta_A)
print(avg_velx_v_A)

avg_velx_v_A = np.mean(dens.values / 16)
print(avg_velx_v_A)

plt.axhline(y = 0, color = 'black', linestyle = '--')
plt.axvline(x = xcut_y, color = 'black', linestyle = '--')

#plt.plot(strip.y, jadj * ez_nontreal * strip.jz / (cc * B0 ** 2), label = f"{jadj} * ezni jz")
#plt.plot(strip.y, ez_nontreal / (cc * B0), label = "$E_{zni}$ / $c B_0$", color = '#d62728')
#plt.axvline(x = strip.y[y_gate_index + 1], color = 'orange', linestyle = '--')

#plt.plot(strip.x, ez_nontreal / strip.jz * 1/10, label = "eta_eff")
#plt.title(f"$\sigma =$ {run.sigma}")

plt.legend(loc = 'lower right')
plt.xlabel('$y$ [$c/\omega_{p}$]')
plt.ylabel('Quantity')
plt.xlim(rect[2:])
plt.ylim([-0.5, 0.5])
plt.show()

In [ ]:
print(0 / 0)

In [ ]:
print(0 / 0)

In [ ]:
print(run.steps)

In [ ]:
flh = FieldLineHelper(run, 16, contour_ds = 1)

rect = [55, 65, 110, 130]
rect = {10: [59.1, 60.9, 162, 167], 0.3: [59, 61, 105, 135]}[run.sigma]
rect = [55, 65, 155, 175]

counter = 0

for ts in range(len(my_steps)):
    plt.close('all')
    cs = fq_cut(d.fields, ts, rect)

    plt.figure()
    plt.title(f"{run.boilerplate} \n t = {run.steps[ts]}")
    
    plot_fq3(cs, cs.by, vabsmax = 5, cbar_label = '$B_y / B_0$', cmap = 'seismic')
    #plot_fq3(cs, cs.vely / v_A, vabsmax = 1.5, cmap = 'seismic', cbar_label = '$v_y / v_A$')
    #plot_streamline(plt, cs, cs.bx, cs.by, 1, 'black') 
    #plot_streamline(plt, cs, cs.velx, cs.vely, 0.5, 'black')
     
    # Downsample the grid for quiver
    step = 5
    xs = cs.x.values[::step]
    ys = cs.y.values[::step]
    X, Y = np.meshgrid(xs, ys)
    U = cs.velx.values[::step, ::step]
    V = cs.vely.values[::step, ::step]
    #plt.quiver(X, Y, U, V, color='white', scale=12, width=0.0055,     # Thinner arrows
    #       headlength=5,     # Optional: tweak arrowhead size
    #       headwidth=3)
    #flh.plot_field_lines(cs, cs.ez[0, 0], arrow_spacing = -1, color = 'white') 
 
    plt.contour(cs.x, cs.y, cs.dens, [2.5], colors = 'green', label = 'n = 2.5')
    lim = 0.2
    plt.contour(cs.x, cs.y, cs.by, [-lim, lim], colors = 'black', label = f'B_y = +- {lim}')
    
    plt.xlim(rect[:2])
    plt.ylim(rect[2:])

    plt.legend()
    
    save_movie_frame(counter) 
    counter += 1

In [ ]:
counter = 0
#flh = FieldLineHelper(run, 8, contour_ds = 4)

j0 = cc * run.B0 / run.c_omp
rect = [40, 80, 100, 155]

for ts in range(len(my_steps))[:1]:#[:5]:
    
    plt.close('all')
    plt.figure()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 5))
    plt.suptitle(f"{run.boilerplate} \n t = {run.steps[0] + ts}")
    cs = fq_cut(d.fields, ts, rect)

    # Left plot: current
    mesh1 = plot_fq3(cs, cs.bx / run.B0, plot=ax1, vabsmax=1.5, cmap='seismic')
    cbar1 = fig.colorbar(mesh1, ax=ax1)
    ax1.set_title('$B_x / B_0$')
    #flh.plot_field_lines(cs, cs.ez[0, 0], plot = ax1, arrow_spacing = -1, color = 'black')

    # Right plot: density
    mesh2 = plot_fq3(cs, cs.dens, cmap='viridis', plot=ax2, vabsmax=5, positive = True)
    cbar2 = fig.colorbar(mesh2, ax=ax2)
    ax2.set_title('$n / n_0$')
    #flh.plot_field_lines(cs, 0, plot = ax2, arrow_spacing = -1, color = 'white')

    plt.tight_layout()

    #flh.plot_field_lines(cs, cs.ez[0, 0], arrow_spacing = -1, color = 'black') 

    for ax in [ax1, ax2]:
        ax.set_xlim(rect[:2])
        ax.set_ylim(rect[2:])
    
    save_movie_frame(counter) 
    counter += 1

    plt.show()

In [ ]:
counter = 0
#flh = FieldLineHelper(run, 8, contour_ds = 4)

j0 = cc * run.B0 / run.c_omp
rect = {26: [40, 80, 100, 155], 67: [50, 70, 145, 195]}[findex]

for ts in range(len(my_steps)):#[:5]:
    
    plt.close('all')
    plt.figure()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 5))
    plt.suptitle(f"{run.boilerplate} \n t = {run.steps[0] + ts}")
    cs = fq_cut(d.fields, ts, rect)

    # Left plot: current
    mesh1 = plot_fq3(cs, cs.jz / j0, plot=ax1, vabsmax=1.5, cmap='seismic')
    cbar1 = fig.colorbar(mesh1, ax=ax1)
    ax1.set_title('$j_z / j_0$')
    #flh.plot_field_lines(cs, cs.ez[0, 0], plot = ax1, arrow_spacing = -1, color = 'black')

    # Right plot: density
    mesh2 = plot_fq3(cs, cs.dens, cmap='viridis', plot=ax2, vabsmax=5, positive = True)
    cbar2 = fig.colorbar(mesh2, ax=ax2)
    ax2.set_title('$n / n_0$')
    ax2.contour(cs.x, cs.y, cs.bx / run.B0, [-0.04, 0.04], colors = 'white')
    #flh.plot_field_lines(cs, 0, plot = ax2, arrow_spacing = -1, color = 'white')

    plt.tight_layout()

    #flh.plot_field_lines(cs, cs.ez[0, 0], arrow_spacing = -1, color = 'black') 

    for ax in [ax1, ax2]:
        ax.set_xlim(rect[:2])
        ax.set_ylim(rect[2:])
    
    save_movie_frame(counter) 
    counter += 1

    #plt.show()

In [ ]:
print(sign_change_indices)

In [ ]:
# X-strip 

strip = xstrip

ez_ideal = strip.bx * strip.vely - strip.by * strip.velx 
ez_nonideal = strip.ez - ez_ideal 
ez_tot = strip.ez
dens = strip.dens1 + strip.dens2

JdotE = strip.jx * strip.ex + strip.jy * strip.ey + strip.jz * strip.ez

esq, bsq, chi = sqrd(strip)


jadj = 200

plt.figure()
#plt.title(boilerplate + f"t = {tchosen}, y = {xcut_y}")

#plt.plot(strip.x, ez_nontreal / (cc * B0), label = "$E_{z, ni} / c B_0$", color = '#d62728')
#plt.plot(strip.x, ez_treal / (cc * B0), label = "ez_treal / c B_0")
#plt.plot(strip.x, strip.ez / (cc * B0), label = "$E_z / c B_0$", color = '#1f77b4') #ez
#plt.plot(strip.x, bsq / B0 ** 2, label = "bsq / B0^2") #bsq
#plt.plot(strip.x, strip.jz / B0 * run.c_omp, label = '$J_z (c / \omega_p) / B_0$', color = '#2ca02c') #jz

#plt.plot(strip.x, strip.by / B0, label = "$B_y / B_0$", color = '#ff7f0e')
#plt.plot(strip.x, strip.epsilon, label = "$\epsilon$")
#plt.plot(strip.x, np.sqrt(bsq) / B0, label = "$B / B_0$", color = '#ff7f0e')
#plt.plot(strip.x, bsq / B0 ** 2, label = "$B^2 / B_0^2$", color = '#ff7f0e')
#plt.plot(strip.x, strip.bz / B0, label = "Bz / B0")
#plt.plot(strip.x, chi, label = '$\chi$')
#plt.plot(strip.x, chi_adj, label = '$\chi_{adj}$')
#plt.plot(strip.x, strip.velx / v_A, label = "$v_x / v_A$", color = '#9467bd')
#plt.plot(strip.x, strip.vely / v_A, label = "$v_y / v_A$", color = 'magenta')
#plt.plot(strip.x, np.sqrt(strip.velx ** 2 + strip.vely ** 2) / v_A, label = "$v / v_A$", color = '#9467bd')
plt.axhline(y = 0, color = 'black', linestyle = '--')
plt.axvline(x = ycut_x, color = 'black', linestyle = '--')

#plt.plot(strip.x, strip.pb, label = "$p_B$")
#plt.plot(strip.x, strip.px, label = "$p_x$")

#plt.plot(strip.x, strip.pb + strip.px, label = "total pressure in x dir")

nbx = strip.bx / np.sqrt(bsq)
nby = strip.by / np.sqrt(bsq)
nbz = strip.bz / np.sqrt(bsq)
#ppara = strip.px * nbx + strip.py * nbx + strip.pz * nbz

#e_cr = (5 * strip.p + 2) / (5 * strip.p + 5)
#epsilon = 1 - 0.5 * (2 * ppara - strip.p)

#plt.plot(strip.x, e_cr, label = "$e_{cr}$")
#plt.plot(strip.x, epsilon, label = "$\epsilon$")

#plt.plot(strip.x, strip.px / strip.pb, label = "$beta$")

btot = np.sqrt(strip.bx ** 2 + strip.by ** 2 + strip.bz ** 2)

# New
#plt.plot(strip.x, strip.ex / (cc * B0), label = "$E_x / c B_0$")
#plt.plot(strip.x, strip.velz / v_A, label = "velz / v_A")
#plt.plot(strip.x, strip.vely / v_A, label = "vely / v_A")
#plt.plot(strip.x, strip.velx / v_A, label = "velx / v_A")
#plt.plot(strip.x, strip.bx / run.B0, label = 'bx / B0')
#plt.plot(strip.x, strip.by / run.B0, label = 'by / B0')
#plt.plot(strip.x, btot / run.B0, label = 'btot')
#plt.plot(strip.x, (strip.ez * strip.bx / btot ** 2) / run.beta_A , label =  'ecrossb over vA')
plt.plot(strip.x, strip.ez, label = 'ez')
#plt.plot(strip.x, -strip.by * strip.velz / (cc * B0), label = "E_xni")
#plt.plot(strip.x, strip.dens1 / 16, label = 'n(e-)')
#plt.plot(strip.x, strip.dens2 / 16, label = 'n(e+)')
#plt.plot(strip.x, strip.dens, label = 'n')
#plt.plot(strip.x, (strip.dens1 + strip.dens2)/ 16, label = '$n$')
#plt.plot(strip.x, (strip.dens1 + strip.dens2)/ 16 * (np.sqrt(bsq) / B0), label = '$(n / n_0) * (B / B_0)$', color = 'grey')
#plt.plot(strip.x, jadj * strip.jz / (dens * B0) * 16, label = f'{jadj} * j / (n / n0)')

#plt.plot(strip.x, (strip.dens2 - strip.dens1) / 16, label = 'rho')
#plt.plot(strip.x, jadj * strip.jx / B0, label = f'{jadj} * jx / B0')

#plt.plot(strip.x, ez_nonideal / strip.jz * 1/10, label = "eta_eff")
#plt.title(f"$\sigma =$ {run.sigma}")
plt.title("" + f"x strip y = {xcut_y}")
plt.xlabel('$x$ [$c/\omega_{p}$]')
plt.ylabel('Quantity')
plt.legend(loc='upper left')

plt.show()

In [ ]:
que = M_SM.values

In [ ]:
plt.figure()
plt.plot(strip.x, M_SM, label="M_SM")
plt.axhline(y=0, color='black', linestyle='--')
plt.axvline(x=ycut_x, color='black', linestyle='--')
plt.title(boilerplate + f" x strip y = {xcut_y}")
plt.xlabel('$x$ [$c/\\omega_{p}$]')
plt.ylabel('M_SM')
plt.legend(loc='center left')
plt.show()

In [ ]:
# Check if any element in in_sqrt is negative
negative_indexer = (in_sqrt < 0).compute()
negative_elements = in_sqrt[negative_indexer]

if negative_elements.size > 0:
    print("There are negative elements in in_sqrt.")
else:
    print("There are no negative elements in in_sqrt.")

In [ ]:
from scipy.integrate import simps

# Define the interval for integration
x_start, x_end = 332, 418
x_start, x_end = 340, 410


# Extract the relevant portion of the data
interval = (strip.x >= x_start) & (strip.x <= x_end)
x_values = strip.x[interval]
bsq_values = bsq[interval] / B0 ** 2

# Perform the integration
average_result = simps(bsq_values, x_values) / (x_end - x_start)

print(f"Integral of B^2 / B_0^2 over the interval [{x_start}, {x_end}] is {average_result}")

In [ ]:
print(list(strip.data_vars))

In [ ]:
print(y_gate[-1][-1])

In [ ]:
p